[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pyshka501/rl-textbook/blob/main/notebooks/ch05_monte_carlo.ipynb)

<div style="background: linear-gradient(135deg, #1a0a28 0%, #2d1b4e 50%, #4a2a7a 100%); padding: 40px; border-radius: 12px; margin-bottom: 20px;">
  <h1 style="color: #c792ea; font-size: 2.2em; margin: 0 0 10px 0;">Chapter 5: Monte Carlo Methods</h1>
  <p style="color: #a8dadc; font-size: 1.1em; margin: 0;">First-Visit MC Prediction, MC Control (epsilon-soft), and Off-Policy MC with Importance Sampling on Blackjack-v1.</p>
</div>

<div style="background: #1e3a5f; padding: 15px; border-radius: 8px; border-left: 4px solid #4fc3f7;">
  <strong style="color: #4fc3f7;">Environment Setup</strong><br>
  <span style="color: #cdd9e5;">Requires <code>gymnasium</code>. Estimated runtime: ~2 minutes on Colab free CPU.</span>
</div>

In [ ]:
!pip install gymnasium numpy matplotlib --quiet

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from collections import defaultdict
import gymnasium as gym

SEED = 42
np.random.seed(SEED)

plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor': '#161b22',
    'axes.edgecolor': '#30363d',
    'axes.labelcolor': '#c9d1d9',
    'xtick.color': '#8b949e',
    'ytick.color': '#8b949e',
    'text.color': '#c9d1d9',
    'grid.color': '#21262d',
    'legend.facecolor': '#161b22',
    'legend.edgecolor': '#30363d',
})

# Verify Blackjack environment
env = gym.make('Blackjack-v1', sab=True)
obs, _ = env.reset(seed=SEED)
print(f'Blackjack-v1 loaded. Observation space: {env.observation_space}')
print(f'Action space: {env.action_space}  (0=Stick, 1=Hit)')
print(f'Sample observation (player_sum, dealer_card, usable_ace): {obs}')

<div style="background: #1e3a5f; padding: 15px; border-radius: 8px; border-left: 4px solid #c792ea;">
  <strong style="color: #c792ea;">1. Blackjack Environment</strong>
</div>

**Blackjack-v1** (Sutton & Barto version):
- **State**: `(player_sum, dealer_showing, usable_ace)` — player's card total (4–21), dealer's visible card (1–10), whether player has a usable ace.
- **Actions**: `0` = Stick (stop), `1` = Hit (draw)
- **Reward**: +1 win, 0 draw, -1 loss
- **No discount** ($\gamma = 1$)

MC methods learn from **complete episodes** without a model — just sample trajectories.

In [ ]:
# ── Episode generation helper ──

def generate_episode(env, policy_fn, seed=None):
    """Run one episode and return list of (state, action, reward)."""
    obs, _ = env.reset(seed=seed)
    episode = []
    done = False
    while not done:
        action = policy_fn(obs)
        next_obs, reward, terminated, truncated, _ = env.step(action)
        episode.append((obs, action, reward))
        obs = next_obs
        done = terminated or truncated
    return episode


# Simple stick-at-20 policy (from Sutton & Barto)
def stick_at_20(obs):
    player_sum, _, _ = obs
    return 0 if player_sum >= 20 else 1  # Stick if 20+, else hit


# Demo episode
ep = generate_episode(env, stick_at_20, seed=SEED)
print('Example episode (stick at 20 policy):')
for step, (s, a, r) in enumerate(ep):
    print(f'  Step {step+1}: state={s}  action={["Stick","Hit"][a]}  reward={r}')

<div style="background: #1e3a5f; padding: 15px; border-radius: 8px; border-left: 4px solid #c792ea;">
  <strong style="color: #c792ea;">2. First-Visit MC Prediction</strong>
</div>

We estimate $V^\pi(s)$ for the stick-at-20 policy by averaging returns from the **first visit** to each state across many episodes:
$$V(s) \approx \frac{1}{|\{\text{episodes visiting } s\}|} \sum_{\text{1st visit}} G_t$$

In [ ]:
# ── First-Visit MC Prediction ──  (~30 sec for 500k episodes)

def first_visit_mc_prediction(env, policy_fn, n_episodes=500_000, gamma=1.0):
    returns_sum = defaultdict(float)
    returns_count = defaultdict(int)

    for ep_i in range(n_episodes):
        episode = generate_episode(env, policy_fn)
        G = 0
        visited = set()
        # Process in reverse to accumulate returns
        for s, a, r in reversed(episode):
            G = gamma * G + r
            if s not in visited:
                returns_sum[s]   += G
                returns_count[s] += 1
                visited.add(s)

    V = {s: returns_sum[s] / returns_count[s] for s in returns_sum}
    return V


print('Running First-Visit MC Prediction (500k episodes)...')
V_mc = first_visit_mc_prediction(env, stick_at_20, n_episodes=500_000)
print(f'States visited: {len(V_mc)}')
# Sample values
for s in [(18, 3, False), (20, 5, False), (20, 1, True)]:
    print(f'  V{s} = {V_mc.get(s, float("nan")):.4f}')

In [ ]:
# ── 3D Value Surface Plot ──

def plot_value_surface(V, usable_ace, title, ax):
    player_sums = np.arange(12, 22)
    dealer_cards = np.arange(1, 11)
    Z = np.zeros((len(player_sums), len(dealer_cards)))
    for i, ps in enumerate(player_sums):
        for j, dc in enumerate(dealer_cards):
            Z[i, j] = V.get((ps, dc, usable_ace), 0.0)
    X, Y = np.meshgrid(dealer_cards, player_sums)
    surf = ax.plot_surface(X, Y, Z, cmap='RdYlGn', alpha=0.85, linewidth=0)
    ax.set_xlabel('Dealer Showing', labelpad=8)
    ax.set_ylabel('Player Sum', labelpad=8)
    ax.set_zlabel('Value', labelpad=8)
    ax.set_title(title, fontsize=10)
    return surf


fig = plt.figure(figsize=(14, 5))
fig.patch.set_facecolor('#0d1117')

ax1 = fig.add_subplot(121, projection='3d', facecolor='#161b22')
ax2 = fig.add_subplot(122, projection='3d', facecolor='#161b22')

plot_value_surface(V_mc, False, 'V^π — No Usable Ace (500k eps)', ax1)
plot_value_surface(V_mc, True,  'V^π — Usable Ace (500k eps)',    ax2)

fig.suptitle('First-Visit MC: Stick-at-20 Policy Value Surface', fontsize=13, color='#c9d1d9')
plt.tight_layout()
plt.show()

<div style="background: #1e3a5f; padding: 15px; border-radius: 8px; border-left: 4px solid #c792ea;">
  <strong style="color: #c792ea;">3. MC Control (ε-soft Policy)</strong>
</div>

MC Control improves the policy at the same time as evaluating it. We maintain an $\varepsilon$-soft policy and update $Q(s,a)$ estimates after each episode, then greedily improve:
$$\pi(a|s) = \begin{cases} 1 - \varepsilon + \varepsilon/|\mathcal{A}| & a = \arg\max_{a'} Q(s,a') \\ \varepsilon/|\mathcal{A}| & \text{otherwise} \end{cases}$$

In [ ]:
# ── MC Control (ε-soft, on-policy) ──  (~60 sec)

def mc_control_epsilon_soft(env, n_episodes=500_000, epsilon=0.1, gamma=1.0):
    Q = defaultdict(lambda: np.zeros(env.action_space.n))
    N = defaultdict(lambda: np.zeros(env.action_space.n))
    wins = []

    def policy(obs):
        if np.random.rand() < epsilon:
            return env.action_space.sample()
        return int(np.argmax(Q[obs]))

    for ep_i in range(n_episodes):
        episode = generate_episode(env, policy)
        G = 0
        sa_visited = set()
        for s, a, r in reversed(episode):
            G = gamma * G + r
            if (s, a) not in sa_visited:
                N[s][a] += 1
                Q[s][a] += (G - Q[s][a]) / N[s][a]
                sa_visited.add((s, a))

        wins.append(episode[-1][2])  # Final reward (+1 win, 0 draw, -1 loss)

    # Extract greedy policy
    greedy_policy = {s: np.argmax(q) for s, q in Q.items()}
    return Q, greedy_policy, wins


print('Running MC Control (500k episodes)...')
Q_mc, pi_mc, win_history = mc_control_epsilon_soft(env, n_episodes=500_000)

# Win rate over time (smoothed)
win_arr = np.array(win_history)
window = 5000
win_rate = np.convolve((win_arr + 1) / 2, np.ones(window)/window, mode='valid')
print(f'Final win rate (last 50k): {win_arr[-50000:].mean()*50+50:.1f}%')
print(f'Unique states with Q values: {len(Q_mc)}')

In [ ]:
# ── Plot learning curve + optimal policy heatmap ──
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 1. Win rate learning curve
ax = axes[0]
ax.plot(np.linspace(0, 500_000, len(win_rate)), win_rate,
        color='#c792ea', linewidth=1.5)
ax.axhline(0.5, color='#8b949e', linestyle='--', alpha=0.6, label='50% baseline')
ax.set_title('MC Control: Win Rate over Time', fontsize=12)
ax.set_xlabel('Episode')
ax.set_ylabel('Win Rate (5k-ep avg)')
ax.set_ylim(0.3, 0.7)
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Optimal policy (no usable ace)
for ax_idx, usable_ace, title in [(1, False, 'Policy: No Usable Ace'), (2, True, 'Policy: Usable Ace')]:
    ax = axes[ax_idx]
    player_sums  = np.arange(12, 22)
    dealer_cards = np.arange(1, 11)
    policy_grid  = np.zeros((len(player_sums), len(dealer_cards)))
    for i, ps in enumerate(player_sums):
        for j, dc in enumerate(dealer_cards):
            state = (ps, dc, usable_ace)
            policy_grid[i, j] = pi_mc.get(state, 1)  # Default: Hit
    ax.imshow(policy_grid, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
    ax.set_yticks(range(len(player_sums)))
    ax.set_yticklabels(player_sums)
    ax.set_xticks(range(len(dealer_cards)))
    ax.set_xticklabels(['A'] + list(range(2, 11)))
    ax.set_xlabel('Dealer Showing')
    ax.set_ylabel('Player Sum')
    ax.set_title(f'{title}\n(Green=Stick, Red=Hit)', fontsize=10)

plt.suptitle('MC Control (ε-soft): Optimal Blackjack Policy', fontsize=14)
plt.tight_layout()
plt.show()

<div style="background: #1e3a5f; padding: 15px; border-radius: 8px; border-left: 4px solid #c792ea;">
  <strong style="color: #c792ea;">4. Off-Policy MC with Importance Sampling</strong>
</div>

**Off-policy** learning separates the **target policy** $\pi$ (to be optimised) from the **behaviour policy** $b$ (used for data collection). The importance sampling ratio corrects for the distribution mismatch:
$$\rho_{t:T-1} = \prod_{k=t}^{T-1} \frac{\pi(A_k|S_k)}{b(A_k|S_k)}$$

**Ordinary IS**: $V(s) = \frac{\sum_j \rho_j G_j}{n}$ — unbiased but high variance.

**Weighted IS**: $V(s) = \frac{\sum_j \rho_j G_j}{\sum_j \rho_j}$ — biased but lower variance.

In [ ]:
# ── Off-Policy MC (Ordinary vs Weighted IS) ──
# Behaviour: random policy (uniform)
# Target: stick at 20

TARGET_STATE = (13, 2, True)   # Evaluate V for this specific state

def off_policy_mc_is(env, target_policy_fn, n_episodes=200_000, gamma=1.0):
    """
    Estimate V^pi(TARGET_STATE) using random behaviour policy + importance sampling.
    Returns arrays of running estimates for ordinary and weighted IS.
    """
    n_actions = env.action_space.n
    ordinary_estimates = []
    weighted_estimates = []

    G_sum     = 0.0
    W_sum     = 0.0
    WG_sum    = 0.0
    count     = 0

    for _ in range(n_episodes):
        # Use random behaviour policy
        episode = generate_episode(env, lambda s: env.action_space.sample())

        # Check if TARGET_STATE is first visited in this episode
        first_idx = None
        for i, (s, a, r) in enumerate(episode):
            if s == TARGET_STATE:
                first_idx = i
                break

        if first_idx is None:
            continue

        # Compute return G from first visit
        G = sum(r for _, _, r in episode[first_idx:])

        # Importance sampling ratio from first_idx onward
        rho = 1.0
        for s, a, _ in episode[first_idx:]:
            pi_a  = float(target_policy_fn(s) == a)  # Deterministic target
            b_a   = 1.0 / n_actions                  # Uniform behaviour
            rho  *= pi_a / b_a
            if rho == 0:
                break

        count  += 1
        G_sum  += G
        W_sum  += rho
        WG_sum += rho * G

        ordinary_estimates.append(G_sum / count if rho > 0 else (ordinary_estimates[-1] if ordinary_estimates else 0))
        weighted_estimates.append(WG_sum / W_sum if W_sum > 0 else 0.0)

    return ordinary_estimates, weighted_estimates


np.random.seed(SEED)
print('Running Off-Policy MC IS (200k episodes)...')
ord_est, wgt_est = off_policy_mc_is(env, stick_at_20, n_episodes=200_000)

true_val = V_mc.get(TARGET_STATE, None)
print(f'Target state: {TARGET_STATE}')
print(f'On-policy MC estimate (reference): {true_val:.4f}')
print(f'Ordinary IS estimate (final):      {ord_est[-1]:.4f}')
print(f'Weighted IS estimate (final):      {wgt_est[-1]:.4f}')

In [ ]:
# ── Plot IS convergence ──
fig, ax = plt.subplots(figsize=(10, 4))

x = np.arange(1, len(ord_est)+1)
ax.plot(x, ord_est, color='#e94560', alpha=0.7, linewidth=1.2, label='Ordinary IS')
ax.plot(x, wgt_est, color='#56d364', alpha=0.9, linewidth=1.8, label='Weighted IS')
if true_val is not None:
    ax.axhline(true_val, color='#ffd700', linestyle='--', linewidth=1.5,
               label=f'On-policy reference ({true_val:.3f})')

ax.set_title(f'Off-Policy MC: V estimate for state {TARGET_STATE}', fontsize=13)
ax.set_xlabel('Episodes visiting this state')
ax.set_ylabel('V estimate')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xscale('log')
plt.tight_layout()
plt.show()

<div style="background: #0d2137; padding: 20px; border-radius: 8px; border: 1px solid #1f6feb; margin-top: 20px;">
  <h3 style="color: #79c0ff; margin-top: 0;">Chapter 5 — Key Takeaways</h3>
  <ul style="color: #c9d1d9; line-height: 1.8;">
    <li><strong>MC methods</strong> learn from complete episodes — no model, no bootstrapping.</li>
    <li><strong>First-Visit MC</strong> averages returns from the first time each state is visited per episode.</li>
    <li><strong>MC Control (ε-soft)</strong> interleaves evaluation and improvement, converging to the ε-optimal policy.</li>
    <li><strong>Off-Policy MC</strong> uses importance sampling to evaluate/improve a different policy than the one generating data.</li>
    <li><strong>Weighted IS</strong> has lower variance than Ordinary IS and is preferred in practice.</li>
  </ul>
</div>